# Qwen3.8-27B → IQ1_M GGUF Quantization (AMD MI300X)

Pipeline: download HF model → build llama.cpp (HIP) → patch MTP loader + quantizer → convert to F16 GGUF → iMatrix → detached IQ1_M quantization → verify + push to Hugging Face.

## Cell 1 — Environment + directories

In [ ]:
from pathlib import Path
import subprocess, os, gc, torch

ROOT = Path("/workspace/qwen38_27b_1bit")
LLAMA_DIR = Path("/workspace/llama.cpp")

for d in [
    ROOT/"hf_model", ROOT/"gguf", ROOT/"calibration",
    ROOT/"quantized", ROOT/"logs", ROOT/"results"
]:
    d.mkdir(parents=True, exist_ok=True)

print("Python:", subprocess.check_output(["python","--version"], text=True).strip())
print("PyTorch:", torch.__version__)
print("ROCm:", torch.version.hip)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/2**30,2), "GB")

## Cell 2 — Install/authenticate Hugging Face

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "huggingface-hub>=1.5.0,<2.0"
], check=True)

from huggingface_hub import login, whoami

try:
    print("HF:", whoami()["name"])
except Exception:
    print("Run:")
    print("from huggingface_hub import login")
    print("login()")
    login()

## Cell 3 — Download Qwen3.8-27B

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3.8-27B"

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(ROOT/"hf_model"),
)

print("HF model ready:", ROOT/"hf_model")

## Cell 4 — Clone/build llama.cpp

In [ ]:
import subprocess, shutil

if not LLAMA_DIR.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/ggml-org/llama.cpp",
        str(LLAMA_DIR)
    ], check=True)

print(subprocess.check_output(
    ["git", "-C", str(LLAMA_DIR), "rev-parse", "--short", "HEAD"],
    text=True
).strip())

subprocess.run([
    "cmake", "-S", str(LLAMA_DIR), "-B", str(LLAMA_DIR/"build"),
    "-DGGML_HIP=ON",
    "-DGGML_NATIVE=OFF",
    "-DCMAKE_BUILD_TYPE=Release",
], check=True)

subprocess.run([
    "cmake", "--build", str(LLAMA_DIR/"build"),
    "-j", "32"
], check=True)

print("BUILD COMPLETE")

## Cell 5 — Patch Qwen3.8 MTP loader

> If this cell says the source differs, **stop there** rather than building a wrong patch.

In [ ]:
from pathlib import Path

qfile = LLAMA_DIR/"src/models/qwen35.cpp"
src = qfile.read_text()

if "regular decoder tensors" not in src:
    old = """    layer.attn_norm      = create_tensor(LLM_TENSOR_ATTN_NORM, "weight", il);
    layer.attn_post_norm = create_tensor(LLM_TENSOR_ATTN_POST_NORM, "weight", il);

    create_tensor_qkv(layer, LLM_TENSOR_ATTN_Q, LLM_TENSOR_ATTN_K, LLM_TENSOR_ATTN_V, "weight", il);
    layer.wo = create_tensor(LLM_TENSOR_ATTN_OUT, "weight", il);
    layer.attn_q_norm = create_tensor(LLM_TENSOR_ATTN_Q_NORM, "weight", il);
    layer.attn_k_norm = create_tensor(LLM_TENSOR_ATTN_K_NORM, "weight", il);

    layer.ffn_gate = create_tensor(LLM_TENSOR_FFN_GATE, "weight", il);
    layer.ffn_down = create_tensor(LLM_TENSOR_FFN_DOWN, "weight", il);
    layer.ffn_up = create_tensor(LLM_TENSOR_FFN_UP, "weight", il);
"""

    new = """    // MTP decoder tensors are real model weights and must be loaded.
    // Only NextN-specific tensors remain controlled by mtp_flags.
    layer.attn_norm      = create_tensor(LLM_TENSOR_ATTN_NORM, "weight", il, 0);
    layer.attn_post_norm = create_tensor(LLM_TENSOR_ATTN_POST_NORM, "weight", il, 0);

    create_tensor_qkv(layer, LLM_TENSOR_ATTN_Q, LLM_TENSOR_ATTN_K, LLM_TENSOR_ATTN_V, "weight", il, 0);
    layer.wo = create_tensor(LLM_TENSOR_ATTN_OUT, "weight", il, 0);
    layer.attn_q_norm = create_tensor(LLM_TENSOR_ATTN_Q_NORM, "weight", il, 0);
    layer.attn_k_norm = create_tensor(LLM_TENSOR_ATTN_K_NORM, "weight", il, 0);

    layer.ffn_gate = create_tensor(LLM_TENSOR_FFN_GATE, "weight", il, 0);
    layer.ffn_down = create_tensor(LLM_TENSOR_FFN_DOWN, "weight", il, 0);
    layer.ffn_up = create_tensor(LLM_TENSOR_FFN_UP, "weight", il, 0);
"""

    # If exact source differs, use the known previously patched source strategy.
    if old not in src:
        print("WARNING: qwen35 source differs from expected patch location.")
        print("Inspect manually before continuing.")
    else:
        qfile.write_text(src.replace(old, new, 1))
        print("Qwen35 loader patched.")
else:
    print("Qwen35 loader patch already present.")

## Cell 6 — Patch quantizer to preserve blk.64

In [ ]:
qfile = LLAMA_DIR/"src/llama-quant.cpp"
src = qfile.read_text()

marker = '    const std::string name = ggml_get_name(tensor);\n'

patch = '''    const std::string name = ggml_get_name(tensor);

    // Qwen3.8/Qwen3.5 MTP block:
    // Standard llama-imatrix does not execute blk.64, so no importance
    // statistics exist for these tensors. Preserve the MTP block.
    if (arch == LLM_ARCH_QWEN35 && name.rfind("blk.64.", 0) == 0) {
        return false;
    }
'''

if "Standard llama-imatrix does not execute blk.64" not in src:
    if marker not in src:
        raise RuntimeError("Quantizer patch location not found")
    src = src.replace(marker, patch, 1)
    qfile.write_text(src)
    print("Quantizer patched.")
else:
    print("Quantizer patch already present.")

subprocess.run([
    "cmake", "--build", str(LLAMA_DIR/"build"), "-j", "32"
], check=True)

print("REBUILD COMPLETE")

## Cell 7 — Convert HF → F16 GGUF

In [ ]:
import subprocess

CONVERT = LLAMA_DIR/"convert_hf_to_gguf.py"
F16_GGUF = ROOT/"gguf"/"qwen3.8-27b-f16.gguf"

if not F16_GGUF.exists():
    subprocess.run([
        "python", str(CONVERT),
        str(ROOT/"hf_model"),
        "--outfile", str(F16_GGUF),
        "--outtype", "f16",
    ], check=True)

print("F16 GGUF:", F16_GGUF)
print("Size:", round(F16_GGUF.stat().st_size/2**30,2), "GB")

## Cell 8 — Create calibration + iMatrix

In [ ]:
import subprocess

CAL = ROOT/"calibration"/"calibration_small.txt"
IMATRIX = ROOT/"calibration"/"imatrix.gguf"
IMATRIX_BIN = LLAMA_DIR/"build"/"bin"/"llama-imatrix"

if not CAL.exists():
    base = "Explain artificial intelligence and machine learning. "
    CAL.write_text((base * 10000))

if not IMATRIX.exists():
    subprocess.run([
        str(IMATRIX_BIN),
        "-m", str(F16_GGUF),
        "-f", str(CAL),
        "-o", str(IMATRIX),
        "-ngl", "99",
        "-t", "32",
        "-B", "512",
        "--chunks", "1",
    ], check=True)

print("Calibration:", CAL.stat().st_size/1024**2, "MB")
print("iMatrix:", IMATRIX.stat().st_size/1024**2, "MB")

## Cell 9 — Start detached IQ1_M quantization

This is the important cell. It does not keep Python attached to the quantizer.

> **Do not restart the kernel while this process is running.**

In [ ]:
import subprocess, os, signal, textwrap

QUANT = LLAMA_DIR/"build"/"bin"/"llama-quantize"
OUT = ROOT/"quantized"/"qwen3.8-27b-IQ1_M-mixed.gguf"
LOG = ROOT/"logs"/"iq1m_quant.log"
PID = ROOT/"logs"/"iq1m_quant.pid"

script = f"""
#!/bin/bash
set -o pipefail

echo "START $(date)" > "{LOG}"

"{QUANT}" \\
  --imatrix "{IMATRIX}" \\
  "{F16_GGUF}" \\
  "{OUT}" \\
  IQ1_M \\
  32 >> "{LOG}" 2>&1

RC=$?
echo "RETURN_CODE=$RC" >> "{LOG}"
echo "END $(date)" >> "{LOG}"
"""

RUNNER = ROOT/"logs"/"run_iq1m.sh"
RUNNER.write_text(script)
RUNNER.chmod(0o755)

p = subprocess.Popen(
    ["nohup", "bash", str(RUNNER)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)

PID.write_text(str(p.pid))

print("Detached quantization started.")
print("PID:", p.pid)
print("Log:", LOG)
print("Output:", OUT)

## Cell 10 — Verify + push automatically when finished

Run this after quantization has had time to finish.

In [ ]:
import time, os, subprocess
from pathlib import Path

OUT = ROOT/"quantized"/"qwen3.8-27b-IQ1_M-mixed.gguf"
LOG = ROOT/"logs"/"iq1m_quant.log"

print("Output exists:", OUT.exists())

if OUT.exists():
    size_gb = OUT.stat().st_size / 2**30
    print("GGUF size:", round(size_gb, 3), "GB")

    tail = LOG.read_text(errors="ignore")[-3000:] if LOG.exists() else ""
    print(tail)

    if "RETURN_CODE=0" in tail:
        print("\nQUANTIZATION SUCCESSFUL")

        # Verify GGUF can be read by llama-cli.
        CLI = LLAMA_DIR/"build"/"bin"/"llama-cli"

        r = subprocess.run(
            [str(CLI), "-m", str(OUT), "-p", "Say hello in one sentence.",
             "-n", "16", "-ngl", "99"],
            text=True,
            capture_output=True,
            timeout=180,
        )

        print("Runtime return code:", r.returncode)
        print(r.stdout[-1500:])

        # Push to HF.
        from huggingface_hub import HfApi, whoami

        username = whoami()["name"]
        repo_id = f"{username}/Qwen3.8-27B-IQ1_M-MI300X"

        api = HfApi()
        api.create_repo(
            repo_id=repo_id,
            repo_type="model",
            exist_ok=True,
        )

        readme = ROOT/"README.md"
        readme.write_text(f"""---
library_name: llama.cpp
tags:
- qwen
- qwen3.8
- gguf
- iq1_m
- ultra-low-bit
- amd
- mi300x
---

# Qwen3.8-27B IQ1_M Mixed

GGUF quantization experiment for Qwen3.8-27B.

## Quantization

- Main transformer trunk: IQ1_M / llama.cpp mixed quantization
- MTP block `blk.64`: preserved in original F16/F32 precision
- Importance matrix: llama.cpp iMatrix
- Hardware used for validation: AMD Instinct MI300X
- Effective model size should be interpreted from the actual GGUF tensor composition, not as uniformly 1-bit.

This artifact is an experimental quantization release.
""")

        api.upload_file(
            path_or_fileobj=str(OUT),
            path_in_repo=OUT.name,
            repo_id=repo_id,
            repo_type="model",
        )

        api.upload_file(
            path_or_fileobj=str(readme),
            path_in_repo="README.md",
            repo_id=repo_id,
            repo_type="model",
        )

        print("PUSHED:", repo_id)

    else:
        print("\nQuantization is still running or failed.")
        print("Latest log:")
        print(tail)
else:
    print("Quantization still running.")
    if LOG.exists():
        print(LOG.read_text(errors="ignore")[-3000:])

**Note:** Cell 9 is what prevents the previous mistake. The 27B IQ1_M conversion is the long-running operation; it is detached from the notebook instead of tying up the kernel.